In [1]:
!pip install causal-conv1d mamba-ssm transformers --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 2.0 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 44.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 21.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 64.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 106.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 22.7 

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoTokenizer
from mamba_ssm import Mamba
from tqdm import tqdm

# ==========================================
# 1. Model Architecture
# ==========================================
class VisionMambaEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384, depth=8):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm': nn.LayerNorm(embed_dim),
                'mamba_fwd': Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2),
                'mamba_bwd': Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
            }) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        
        self.apply(self._init_weights)
        # Truncated normal for positional embeddings
        nn.init.normal_(self.pos_embed, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2) 
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) 
        x = x + self.pos_embed
        
        for layer in self.layers:
            normed = layer['norm'](x)
            fwd = layer['mamba_fwd'](normed)
            bwd = layer['mamba_bwd'](normed.flip([1])).flip([1])
            x = x + fwd + bwd
            
        return self.norm(x)

class VisionMambaCaptioner(nn.Module):
    def __init__(self, vocab_size, embed_dim=384, encoder_depth=8, decoder_depth=4):
        super().__init__()
        self.encoder = VisionMambaEncoder(embed_dim=embed_dim, depth=encoder_depth)
        self.text_emb = nn.Embedding(vocab_size, embed_dim)
        
        self.decoder_layers = nn.ModuleList([
            nn.ModuleDict({
                'norm': nn.LayerNorm(embed_dim),
                'mamba': Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
            }) for _ in range(decoder_depth)
        ])
        
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.lm_head.weight = self.text_emb.weight # Tie weights
        self.num_vis_tokens = self.encoder.num_patches + 1

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def forward(self, images, input_ids):
        vis_tokens = self.encoder(images)          
        txt_tokens = self.text_emb(input_ids)      
        
        x = torch.cat([vis_tokens, txt_tokens], dim=1) 
        
        for layer in self.decoder_layers:
            x = x + layer['mamba'](layer['norm'](x))
            
        txt_out = x[:, self.num_vis_tokens - 1 : -1, :] 
        return self.lm_head(txt_out)
# ==========================================
# 2. Dataset Setup
# ==========================================

class COCODataset(Dataset):
    def __init__(self, img_dir, ann_file, tokenizer, transform=None):
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        
        with open(ann_file, 'r') as f:
            data = json.load(f)
            
        self.images = {img['id']: img['file_name'] for img in data['images']}
        self.annotations = data['annotations']
        
    def __len__(self):
        return len(self.annotations)
        
    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_id = ann['image_id']
        caption = ann['caption']
        
        # COCO filenames are zero-padded 12-digit integers
        filename = str(img_id).zfill(12) + '.jpg'
        if filename not in self.images.values():
            filename = self.images[img_id] 
            
        img_path = os.path.join(self.img_dir, filename)
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        tokens = self.tokenizer(
            caption + self.tokenizer.eos_token, 
            truncation=True, 
            max_length=64, 
            return_tensors='pt'
        )['input_ids'].squeeze(0)
        
        return image, tokens
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if __name__=="__main__":
    import os
    
    # 1. Set seed globally
    set_seed(42)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    assert device.type == 'cuda', "Mamba heavily requires CUDA to run efficiently!"

    IMG_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017/"
    ANN_FILE = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Rest of Dataset & DataLoader setup remains identical...
    dataset = COCODataset(IMG_DIR, ANN_FILE, tokenizer, transform)
    
    def collate_fn(batch):
        images, tokens_list = zip(*batch)
        images = torch.stack(images, dim=0)
        max_len = max(len(t) for t in tokens_list)
        padded = torch.full((len(tokens_list), max_len), tokenizer.pad_token_id, dtype=torch.long)
        for i, t in enumerate(tokens_list):
            padded[i, :len(t)] = t
        return images, padded
        
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True, 
                            collate_fn=collate_fn, num_workers=4, pin_memory=True)

    model = VisionMambaCaptioner(
        vocab_size=len(tokenizer), 
        embed_dim=384,       
        encoder_depth=8,     
        decoder_depth=4      
    ).to(device)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {num_params / 1e6:.2f}M")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    
    # 2. Updated GradScaler for newer PyTorch versions
    scaler = torch.amp.GradScaler('cuda') 

    start_epoch = 0
    RESUME_CHECKPOINT = None 
    
    if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
        print(f"Loading checkpoint from {RESUME_CHECKPOINT}")
        checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch + 1}")

    epochs = 5
    for epoch in range(start_epoch, epochs):
        model.train()
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for images, tokens in pbar:
            images, tokens = images.to(device, non_blocking=True), tokens.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            # 3. Updated autocast to fix FutureWarning
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = model(images, tokens)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tokens.reshape(-1))
                
            scaler.scale(loss).backward()
            
            # 4. Optional but highly recommended: Gradient clipping to prevent early spikes
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"vim_epoch_{epoch+1}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': loss.item(),
        }, checkpoint_path)

Total Trainable Parameters: 38.96M


Epoch 2/5:  48%|████▊     | 4452/9247 [41:25<44:33,  1.79it/s, loss=2.4643]  